# NGSO SLS - Slice A: Coverage Explorer

Interactive coverage/availability analysis for NGSO constellations (analytic Kepler+J2, H3 grid, single-owner sharding). Presets cover the **Reliance-Jio sizing scenarios**; **Custom (Walker)** gives full T/P/F + altitude + inclination for up to two shells.

**k (min sats in view)** = minimum satellites simultaneously above the min elevation for a cell to count as covered; **availability** = fraction of time that holds.

**k=1 make-before-break gate** (optional): tests true handover continuity — a cell passes only if it is continuously served by ≥1 satellite **and** every serving transition has a ≥ *Min 2-sat overlap* window (a real make-before-break handoff). Overlap is measured to ±one time-step, so set *Time step* to order-seconds for second-scale windows.

Each run **appends a new tab** (previous runs are kept for comparison), and **all input parameters are saved into the output CSV** (comment header) and echoed in the run panel.

Two tools: (A) a single-scenario **Coverage Explorer**, and (B) a **minimum-satellite sweep** that finds the smallest constellation meeting a coverage grade over an area. Run cells top to bottom; after pushing new code, Runtime -> Restart runtime -> Run all.

## Setup

In [ ]:
# === Setup: make ngso_sls importable (Colab + local Jupyter Lab) ===
# LOCAL JUPYTER (recommended): in a terminal, `pip install -e .` in the repo once, then this
#   cell is a no-op (it detects ngso_sls and skips clone/install entirely).
# COLAB: set REPO_URL to your remote. Private repo -> add a GitHub token in Colab 'Secrets'
#   named GITHUB_TOKEN (enable Notebook access).
# NOTE: after you push new code, do Runtime -> Restart runtime, then Run all — a running kernel
#   keeps already-imported modules, so code changes only take effect on a fresh kernel.
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"

import importlib, importlib.util, subprocess, sys, os, re


def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) - see output above")


if importlib.util.find_spec("ngso_sls") is None:  # no-op on local Jupyter if already installed
    url = REPO_URL
    try:  # optional private-repo auth via Colab secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _run(f"git clone {url} {repo_dir}")
    else:
        _run(f"git -C {repo_dir} pull --ff-only")  # update a stale clone on a fresh kernel
    _run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)}")
    sys.path.insert(0, os.path.abspath(repo_dir))
    importlib.invalidate_caches()

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

## A. Coverage Explorer — controls
Set parameters here. `Global` at high H3 resolution is heavy - start coarse.

In [ ]:
from ngso_sls.explorer import CoverageExplorer, MinSatSweep
from IPython.display import display

explorer = CoverageExplorer()
display(explorer.controls)

### Coverage Explorer — results
Click **Run simulation** (or run this cell). Each run appends a new **Run N** tab (previous runs kept) with: availability map, mean-sats-in-view map, **MBB feasibility** map (when the k=1 make-before-break gate is on), sats-in-view vs latitude, availability histogram. Each run writes both `coverage_availability.csv` (latest) and a per-run `coverage_availability_run{N}.csv` (with the input parameters in its header). Use the **✕ Close run** button in a tab to remove it — its CSV stays on disk.

In [ ]:
display(explorer.results)
explorer.run()   # initial run; also re-runs when you click "Run simulation" above

## B. Minimum-satellite sweep — controls
Thins a single Walker shell (planes x sats/plane -> total N) and measures area coverage, to find the **smallest constellation** meeting the grade: *X% of area at Y% availability, with k satellites in view*. Each swept N is a full coverage run, so more sweep points / longer duration = longer runtime (progress bar shows it).

In [ ]:
sweep = MinSatSweep()
display(sweep.controls)

### Minimum-satellite sweep — results
Click **Run sweep** (or run this cell). Shows the coverage-vs-N curve with the minimum-N marker, and writes `min_sat_sweep.csv`.

In [ ]:
display(sweep.results)
sweep.run()   # initial sweep; re-runs when you click "Run sweep" above

## Terrain fetch check (optional)
The Coverage Explorer's **"Account for terrain"** toggle uses **ETOPO** (NOAA ERDDAP, no API key) by default, subset to your AOR. To verify the fetch works in your environment, set `VERIFY_TERRAIN = True` below and run — it pulls a small tile over the Everest region and should report a high `elev_max_m`.

In [ ]:
# Optional: verify the ETOPO runtime fetch (ERDDAP). Set True and run.
VERIFY_TERRAIN = False
if VERIFY_TERRAIN:
    from ngso_sls.terrain import fetch_dem_selftest
    print(fetch_dem_selftest())   # Everest region -> expect elev_max_m in the thousands